## Import The Required Libraries

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory, ConversationTokenBufferMemory

In [3]:
load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

## CONVERSATIONBUFFERMEMORY — keeps everything

In [4]:
print("=== ConversationBufferMemory ===")
buffer_memory = ConversationBufferMemory(return_messages=True)

buffer_memory.chat_memory.add_user_message("My name is Rishit and I work in Boston.")
buffer_memory.chat_memory.add_ai_message("Nice to meet you Rishit! Boston is a great tech hub.")
buffer_memory.chat_memory.add_user_message("I'm a backend engineer with 2 years experience.")
buffer_memory.chat_memory.add_ai_message("That's impressive! Backend engineering is in high demand.")

history = buffer_memory.load_memory_variables({})
print(f"Messages stored: {len(history['history'])}")
for msg in history['history']:
    print(f"{type(msg).__name__}: {msg.content[:60]}")
print()



=== ConversationBufferMemory ===
Messages stored: 4
HumanMessage: My name is Rishit and I work in Boston.
AIMessage: Nice to meet you Rishit! Boston is a great tech hub.
HumanMessage: I'm a backend engineer with 2 years experience.
AIMessage: That's impressive! Backend engineering is in high demand.



C:\Users\rishi\AppData\Local\Temp\ipykernel_23380\2026745531.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  buffer_memory = ConversationBufferMemory(return_messages=True)


##  CONVERSATIONBUFFERWINDOWMEMORY — last K exchanges

In [5]:
print("=== ConversationBufferWindowMemory (k=1) ===")
window_memory = ConversationBufferWindowMemory(k=1, return_messages=True)

window_memory.chat_memory.add_user_message("My name is Rishit and I work in Boston.")
window_memory.chat_memory.add_ai_message("Nice to meet you Rishit! Boston is a great tech hub.")
window_memory.chat_memory.add_user_message("I'm a backend engineer with 2 years experience.")
window_memory.chat_memory.add_ai_message("That's impressive! Backend engineering is in high demand.")

history = window_memory.load_memory_variables({})
print(f"Messages stored: {len(history['history'])}")
for msg in history['history']:
    print(f"{type(msg).__name__}: {msg.content[:60]}")
print()

=== ConversationBufferWindowMemory (k=1) ===
Messages stored: 2
HumanMessage: I'm a backend engineer with 2 years experience.
AIMessage: That's impressive! Backend engineering is in high demand.



C:\Users\rishi\AppData\Local\Temp\ipykernel_23380\446062323.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=1, return_messages=True)


## CONVERSATIONSUMMARYMEMORY — compresses old messages

In [6]:
print("=== ConversationSummaryMemory ===")
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)

summary_memory.save_context(
    {"input": "I'm building a SaaS API with FastAPI and PostgreSQL."},
    {"output": "That's a solid stack. FastAPI is great for async Python APIs."}
)

summary_memory.save_context(
    {"input": "I'm implementing multi-tenancy with JWT and RBAC."},
    {"output": "Good approach. Make sure every query filters by tenant_id."}
)

history = summary_memory.load_memory_variables({})
print("summary of conversation:")
print(history)
print()

=== ConversationSummaryMemory ===


C:\Users\rishi\AppData\Local\Temp\ipykernel_23380\1484589970.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)


summary of conversation:
{'history': [SystemMessage(content="The human says they're building a SaaS API with FastAPI and PostgreSQL. The AI says it's a solid stack and that FastAPI is great for async Python APIs. The human mentions implementing multi-tenancy with JWT and RBAC. The AI agrees it's a good approach and adds that every query should filter by tenant_id.", additional_kwargs={}, response_metadata={})]}



## CONVERSATIONTOKENBUFFERMEMORY — token-based trimming 

In [7]:
print("=== ConversationTokenBufferMemory ===")
token_memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50, return_messages=True)

token_memory.save_context(
    {"input": "Hello, I am building a backend project."},
    {"output": "Great! What kind of project?"}
)

token_memory.save_context(
    {"input": "A SaaS API with multi-tenancy."},
    {"output": "Interesting. What tech stack are you using?"}
)

history = token_memory.load_memory_variables({})
print(f"Messages stored: {len(history['history'])}")
for msg in history['history']:
    print(f"{type(msg).__name__}: {msg.content}")
print()


=== ConversationTokenBufferMemory ===


C:\Users\rishi\AppData\Local\Temp\ipykernel_23380\3624194220.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  token_memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50, return_messages=True)


Messages stored: 3
AIMessage: Great! What kind of project?
HumanMessage: A SaaS API with multi-tenancy.
AIMessage: Interesting. What tech stack are you using?



## MANUAL CHAT LOOP WITH MESSAGESSPLACHEHOLDER — the modern LCEL way

In [8]:
print("=== Manual Chat Loop with MessagesPlaceholder ===")

chat_template = ChatPromptTemplate.from_messages([
    ("system",  "You are a helpful backend engineering coach. Be concise."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = chat_template | llm | StrOutputParser()

chat_history: list[HumanMessage | AIMessage] = []

def chat(input: str) -> str:
    """Single turn chat function that maintains conversation history."""
    chat_history.append(HumanMessage(content=input))
    response = chain.invoke({
        "history": chat_history,
        "input": input
    })
    chat_history.append(AIMessage(content=response))
    return response

r1 = chat("My name is Rishit and I'm preparing for backend engineering interviews.")
print(f"Turn 1 response: {r1[:100]}")

r2 = chat("What should I focus on for system design questions?")
print(f"Turn 2 response: {r2[:100]}")

r3 = chat("What's my name again?")
print(f"Turn 3 response: {r3[:100]}")

print(f"\nTotal messages in history: {len(chat_history)}")

print(f"History contains: {[(type(msg).__name__, msg.content[:60]) for msg in chat_history]}")

=== Manual Chat Loop with MessagesPlaceholder ===
Turn 1 response: Nice to meet you, Rishit. I can help you prep for backend interviews. To tailor a plan, quick questi
Turn 2 response: Great question. Here’s a focused way to approach system design questions in interviews.

What to foc
Turn 3 response: Your name is Rishit.

Would you like me to use a nickname or another preferred name?

Total messages in history: 6
History contains: [('HumanMessage', "My name is Rishit and I'm preparing for backend engineering "), ('AIMessage', 'Nice to meet you, Rishit. I can help you prep for backend in'), ('HumanMessage', 'What should I focus on for system design questions?'), ('AIMessage', 'Great question. Here’s a focused way to approach system desi'), ('HumanMessage', "What's my name again?"), ('AIMessage', 'Your name is Rishit.\n\nWould you like me to use a nickname or')]
